### Assisgment:
Create a simple assistant that uses any LLM and should be pydantic, when we ask about any product it should give you two information product Name, product details tentative price in USD (integer). use chat Prompt Template.


In [32]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import Optional

load_dotenv()

os.environ["GROQ_API_KEY"]= os.getenv("GROQ_API_KEY")
os.environ["LANGCHAIN_API_KEY"]= os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"]= os.getenv("LANGCHAIN_PROJECT")
os.environ["LANGCHAIN_TRACING_V2"]= "true"

model=ChatGroq(model="gemma2-9b-it")

class Product(BaseModel):
    productname: str=Field(description="Name of the product")
    details:str=Field(min_length=2,max_length=2000,description= "Describe the product")
    tentativeprice:float= Field(description="Price should be in USD. If unknown, leave empty.")
output_parser=JsonOutputParser(pydantic_object=Product)

prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are an expert Product Manager. Provide me information on the asked products"),
        ("human","Extract the information from the following text:\n\n{input}\n\n{format_instructions}")
    ]
)

prompt=prompt.partial(format_instructions=output_parser.get_format_instructions())

chain=prompt|model|output_parser

response=chain.invoke({"input":"Tell me about iphone 15"})
print(response)


{'productname': 'iPhone 15', 'details': 'The iPhone 15 is the latest flagship smartphone from Apple. It features a new design, an upgraded camera system, and the latest A-series chip. ', 'tentativeprice': 799.99}
